# Validation Test Evaluation - Rising Bubble Benchmark Testcase 1

This notebook and the corresponding simulation setup notebook (RisingBubble_Testcase1_Run.ipynb) are part of published results (cf. 7.3.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853). We compare our results against the rising bubble benchmark test case established by Hysing et al ( https://doi.org/10.1002/fld.1934)

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_RisingBubble");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\XNSEpaper_RisingBubble");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_RisingBubble");

Opening existing database '\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_RisingBubble'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart2*	5/31/2020 8:10:24 PM	97a64a0d...
#1: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart1*	4/17/2020 2:43:50 PM	7a7385d1...
#2: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh40	3/23/2020 10:05:22 PM	44504bf6...
#3: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh20	3/23/2020 10:05:20 PM	d3c1cfe5...
#4: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80*	3/23/2020 10:05:28 PM	931391e0...
#5: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40	3/23/2020 10:43:42 AM	de42008a...
#6: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart5	6/5/2020 9:21:58 PM	6e3cf7d8...
#7: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart4*	6/3/2020 10:12:02 PM	22177ad6...
#8: XNSEpaper_RisingBubble	RisingBubble_tc2_ConvStudy_k2_mesh80_restart3*	6/1/2020 4:00:14 PM	42b883f6...
#9: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh60	3/26/2020 9

# Test case 1 - Ellipsoidal shape

In [5]:
int[] pDegS = { 2, 3 };
int[] Resolutions = { 10, 20, 40, 60, 80 };

In [6]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_RisingBubble");

Error: System.UnauthorizedAccessException: Access to the path '\\fdygitrunner\ValidationTests\databases\XNSEpaper_RisingBubble\bbf8c6e9-3624-41c9-8b7a-f5a13ad6f05a.token' is denied.
   at Microsoft.Win32.SafeHandles.SafeFileHandle.CreateFile(String fullPath, FileMode mode, FileAccess access, FileShare share, FileOptions options)
   at Microsoft.Win32.SafeHandles.SafeFileHandle.Open(String fullPath, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, Nullable`1 unixCreateMode)
   at System.IO.File.OpenHandle(String path, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize)
   at System.IO.File.WriteToFile(String path, FileMode mode, ReadOnlySpan`1 contents, Encoding encoding)
   at System.IO.File.WriteAllText(String path, String contents, Encoding encoding)
   at BoSSS.Foundation.IO.DatabaseInfo.PathMatch(String otherPath) in C:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\Data

In [6]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int pDeg in pDegS) {
foreach (int Res in Resolutions) {
    if (pDeg == 2 && Res == 10 
        || pDeg == 3 && Res == 80)
        continue; 
    string studyName = $"RisingBubble_tc1_ConvStudy_k{pDeg}_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
evalSess

Searching for: RisingBubble_tc1_ConvStudy_k2_mesh20
found
Searching for: RisingBubble_tc1_ConvStudy_k2_mesh40
found
Searching for: RisingBubble_tc1_ConvStudy_k2_mesh60
found
Searching for: RisingBubble_tc1_ConvStudy_k2_mesh80
found
Searching for: RisingBubble_tc1_ConvStudy_k3_mesh10
found
Searching for: RisingBubble_tc1_ConvStudy_k3_mesh20
Restart session found! Take last run
Searching for: RisingBubble_tc1_ConvStudy_k3_mesh40
found
Searching for: RisingBubble_tc1_ConvStudy_k3_mesh60
found


#0: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh20	3/23/2020 10:43:07 AM	6106f0e5...
#1: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh40	3/23/2020 10:43:12 AM	97e8e806...
#2: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh60	3/26/2020 9:42:42 AM	4510aed9...
#3: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k2_mesh80	3/23/2020 11:06:01 AM	b74d773f...
#4: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50	6/5/2020 8:57:58 PM	fdda4efd...
#5: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh20_restart1	5/9/2020 8:49:47 AM	4c450744...
#6: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh40	3/23/2020 10:43:42 AM	de42008a...
#7: XNSEpaper_RisingBubble	RisingBubble_tc1_ConvStudy_k3_mesh60	3/26/2020 9:43:00 AM	f6cdd826...


In [8]:
NUnit.Framework.Assert.AreEqual(8, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [10]:
bool calcComputeTimes = false;

In [11]:
if (calcComputeTimes) {

foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].ToLower().StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

## Table 5

In [12]:
public static void GetSimulationProperties(List<ISessionInfo> evalSess) {

foreach (var sess in evalSess) {
    Console.WriteLine(sess.Name);

    int NEL = Convert.ToInt32(sess.KeysAndQueries["Grid:NoOfCells"]);
    Console.WriteLine($"NEL = {NEL}");

    int timesteps = sess.Timesteps.Count;
    int numCCmax  = 0;
    for(int ts = 0; ts < timesteps; ts++) {
        
        var phi    = sess.Timesteps.Pick(ts).GetField("Phi");
        var LevSet = new LevelSet(phi.Basis, "LevelSet"); 
        LevSet.Acc(1.0, phi);
        var LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
        LsTrk.UpdateTracker(sess.Timesteps.Pick(ts).PhysicalTime);
        
        int numCC = LsTrk.Regions.GetCutCellMask().Count();
        if(numCC > numCCmax)
            numCCmax = numCC;
    }

    string[] nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    int NDOF = (nameData[3] == "k2") ? (NEL + numCCmax) * 15 : (NEL + numCCmax) * 26;
    Console.WriteLine($"NDOF = {NDOF} (max number of CC = {numCCmax})");

    int NTS = Convert.ToInt32(sess.KeysAndQueries["NoOfTimesteps"]);
    Console.WriteLine($"NTS = {NTS}");
}
}

In [12]:
GetSimulationProperties(evalSess)

RisingBubble_tc1_ConvStudy_k2_mesh20
NEL = 800


Error: Command cancelled.

## Evaluation - rising bubble benchmark quantities

In order to compare the different numerical methods three scalar measures for the temporal evolution of the rising bubble are introduced. 
First, the center of mass is considered given as
>$$
\vec{x}_c = \frac{ \int_{\mathfrak{A}} \vec{x} dV }{ \int_{\mathfrak{A}} 1 dV}.
$$<
The second measure is denoted as circularity and defined by
>$$
\cancel{c} = \frac{\text{perimeter of area-equivalent circle}}{\text{perimeter of bubble}}.
$$<
The circularity is $\cancel{c} = 1$ for a perfectly circular bubble and gets smaller for more deformed bubble shapes. 
The third measure is the mean bubble velocity
>$$
\vec{u}_c = \frac{ \int_{\mathfrak{A}} \vec{u} dV }{ \int_{\mathfrak{A}} 1 dV },
$$< 
where the component in $y$-direction is denoted as the rise velocity $V_c$. The error quantification is done via the $l_2$-error norm (44) and the corresponding ROC values (45). 
Note that in the original benchmark paper furthermore $l_1$  and $l_\infty$-error norms are considered. 
Besides the three scalar quantities the terminal shape of the bubble at t = 3 is compared

In [9]:
evalSess = evalSess.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();

In [10]:
static List<Plot2Ddata> evalData = new List<Plot2Ddata>();

In [11]:
Plot2Ddata riseHeightData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "center of mass - y");
evalData.Add(riseHeightData);

Processing session: RisingBubble_tc1_ConvStudy_k2_mesh80
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh20
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh20_restart1
  Found restart session: 8351c0ba-a604-479c-97b7-5c7496680bad
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50
  Merging data from 1 sessions.
... Done
Build DataSet


In [12]:
Plot2Ddata circData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "circularity");
evalData.Add(circData);

Processing session: RisingBubble_tc1_ConvStudy_k2_mesh80
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh20
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh20_restart1
  Found restart session: 8351c0ba-a604-479c-97b7-5c7496680bad
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50
  Merging data from 1 sessions.
... Done
Build DataSet


In [13]:
Plot2Ddata riseVelocityData = evalSess.ReadLogDataValue("BenchmarkQuantities_RisingBubble", "rise velocity", new string[] { "mean velocity y" } );
evalData.Add(riseVelocityData);

Processing session: RisingBubble_tc1_ConvStudy_k2_mesh80
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh60
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh40
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k2_mesh20
  Merging data from 1 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh20_restart1
  Found restart session: 8351c0ba-a604-479c-97b7-5c7496680bad
  Merging data from 2 sessions.
... Done
Processing session: RisingBubble_tc1_ConvStudy_k3_mesh10_ReInit50
  Merging data from 1 sessions.
... Done
Build DataSet


In [14]:
public static List<Plot2Ddata> GetErrorNormForConvergence_Plot2Ddata(List<Plot2Ddata> data, List<ISessionInfo> dataSess, string study, int pDeg) {

    List<Plot2Ddata> studyData = new List<Plot2Ddata>();
    foreach (var pltDat in data) {
        Plot2Ddata plt = new Plot2Ddata();
        foreach (var dgrp in pltDat.dataGroups) {
            if (dgrp.Name.Contains($"{study}-ConvStudy-k{pDeg}")) {
                plt.AddDataGroup(dgrp);
            }
        }
        studyData.Add(plt);
    }

    double[] abscissa = dataSess.Where(sess => sess.Name.Contains($"{study}_ConvStudy_k{pDeg}")).Select(s => Convert.ToDouble(s.KeysAndQueries["Grid:hMax"])).Skip(1).ToArray();
    var convData = ISessionInfoExtensions.LogDataToConvergenceData(studyData, abscissa);

    return convData;
}

In [15]:
public static void Checkl2errorNorms(List<Plot2Ddata> errNorms, string qIdxdName, int errInd, string[] meshSize, double[] refl2Values, double[] refROC) {

    int qIdx = -1;
    for (int i = 0; i < evalData.Count(); i++) {
        if (evalData.ElementAt(i).Ylabel == qIdxdName)
            qIdx = i;
    }
    Console.WriteLine($"Checking norms for quantity: {qIdxdName} ({qIdx})");

    var norms = errNorms.ElementAt(qIdx).dataGroups[errInd];
    var values = norms.Values.CloneAs().Reverse().ToArray();
    var abscissas = norms.Abscissas.CloneAs().Reverse().ToArray();

    int Lstudy = meshSize.Length;
    for (int i = 0; i < Lstudy; i++) {

        double val = values[i];
        // rate of convergence (ROC)
        double ROC = -1.0;
        if (i > 0) {
            ROC = Math.Log10(values[i-1]/val)/Math.Log10(abscissas[i-1]/abscissas[i]);
        }

        Console.WriteLine($"1/h = {meshSize[i]}: {val}; ROC = {ROC}");

        if (qIdxdName == "circularity" && meshSize[i] == "10") {
            Console.WriteLine("Circularity error on corasest mesh is too unreliable, skip assertion.");
            continue;
        } 
        
        double tolVal = refl2Values[i] * 1.75;
        //Console.WriteLine($"tolerance for value: {tolVal}");
        NUnit.Framework.Assert.LessOrEqual(val, tolVal, 
            $"{norms.Name} for mesh{meshSize[i]} does not match reference value {refl2Values[i]}.");

        if (i > 0 && ROC > 0 ) {

            if (qIdxdName == "center of mass - y" && meshSize[i] == "20") {
                continue;
            } 

            double tolROC = refROC[i] - 0.6;
            //Console.WriteLine($"tolerance for ROC: {tolROC}");
            NUnit.Framework.Assert.GreaterOrEqual(ROC, tolROC, 
                $"Corresponding ROC does not match reference value {refROC[i]}.");
        }
    }
} 

## TABLE 6 - $l_2$-error norms and ROC

In [20]:
var errNorms_k2 = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, "tc1", 2);

Center of mass for $k=2$

In [21]:
Checkl2errorNorms(errNorms_k2, "center of mass - y", 1, new string[] { "20", "40", "60" }, 
    new double[] { 4.92e-3, 1.27e-3, 4.53e-4 }, new double[] { 0.0, 1.95, 2.54 });

Checking norms for quantity: center of mass - y (0)
1/h = 20: 0.004915502501134134; ROC = -1
1/h = 40: 0.001267772910148286; ROC = 1.9550425618452472
1/h = 60: 0.0004527921942505231; ROC = 2.539265938133667


Rise velocity for $k=2$

In [22]:
Checkl2errorNorms(errNorms_k2, "rise velocity", 1, new string[] { "20", "40", "60" }, 
    new double[] { 1.02e-2, 2.95e-3, 1.05e-3 }, new double[] { 0.0, 1.79, 2.55 });

Checking norms for quantity: rise velocity (2)
1/h = 20: 0.010232923993698453; ROC = -1
1/h = 40: 0.0029451599702992066; ROC = 1.796800541824527
1/h = 60: 0.0010467844522646466; ROC = 2.5512431903452932


Circularity for $k=2$

In [23]:
Checkl2errorNorms(errNorms_k2, "circularity", 1, new string[] { "20", "40", "60" }, 
    new double[] { 8.77e-4, 5.49e-4, 1.92e-4 }, new double[] { 0.0, 0.68, 2.59 });

Checking norms for quantity: circularity (1)
1/h = 20: 0.0008773813742428139; ROC = -1
1/h = 40: 0.0005486048477688396; ROC = 0.6774367093166731
1/h = 60: 0.00019151210341910465; ROC = 2.5956053534098094


In [24]:
var errNorms_k3 = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, "tc1", 3);

Center of mass for $k=3$

In [25]:
Checkl2errorNorms(errNorms_k3, "center of mass - y", 1, new string[] { "10", "20", "40" }, 
    new double[] { 1.44e-2, 2.23e-3, 6.85e-4 }, new double[] { 0.0, 2.69, 1.70 });

Checking norms for quantity: center of mass - y (0)
1/h = 10: 0.014399096335809007; ROC = -1
1/h = 20: 0.0029925734690351897; ROC = 2.266519701748366
1/h = 40: 0.0006853531595083737; ROC = 2.1264671671373776


Rise velocity for $k=3$

In [26]:
Checkl2errorNorms(errNorms_k3, "rise velocity", 1, new string[] { "10", "20", "40" }, 
    new double[] { 1.56e-2, 5.96e-3, 1.69e-3 }, new double[] { 0.0, 1.39, 1.82 });

Checking norms for quantity: rise velocity (2)
1/h = 10: 0.015618808516626928; ROC = -1
1/h = 20: 0.007290637846146833; ROC = 1.0991674572004766
1/h = 40: 0.0016899439667427208; ROC = 2.1090696270918565


Circularity for $k=3$

In [27]:
Checkl2errorNorms(errNorms_k3, "circularity", 1, new string[] { "10", "20", "40" }, 
    new double[] { 9.58e-4, 1.65e-3, 4.67e-4 }, new double[] { 0.0, -0.78, 1.04 });

Checking norms for quantity: circularity (1)
1/h = 10: 0.0009570067365552281; ROC = -1
Circularity error on corasest mesh is too unreliable, skip assertion.
1/h = 20: 0.0015756159438916073; ROC = -0.7193149357150421
1/h = 40: 0.00046746907676203296; ROC = 1.7529730824806768


## TABLE B1 - $l_1$-error norms and ROC

## TABLE B2 - $l_\infty$-error norms and ROC

## TABLE B3 - Benchmark quantities at distinct values in time for the finest solutions of the respective groups

### Minimum circularity and instance of time

In [16]:
public static (double circMin, double time) GetMinimumCircularity(Plot2Ddata.XYvalues circData) {
    double circMin = circData.Values[0];
    int minIndex = 0;
    for (int i = 1; i < circData.Values.Length; ++i) {
        if (circData.Values[i] < circMin) {
            circMin = circData.Values[i];
            minIndex = i;
        }
    }

    return (circMin, circData.Abscissas[minIndex]);
}

### Maximum rise velocity and instance of time

In [17]:
public static (double riseVelMax, double time) GetMaximumRiseVelocity(Plot2Ddata.XYvalues riseData) {
    double riseMax = riseData.Values[0];
    int maxIndex = 0;
    for (int i = 1; i < riseData.Values.Length; ++i) {
        if (riseData.Values[i] > riseMax) {
            riseMax = riseData.Values[i];
            maxIndex = i;
        }
    }

    return (riseMax, riseData.Abscissas[maxIndex]);
}

### Terminal rise height

In [18]:
public static double GetTerminalRiseHeight(Plot2Ddata.XYvalues yData) {
    return yData.Values.Last();
}

In [19]:
public static void CheckBenchmarkQuantities(List<Plot2Ddata> data, string study, int pDeg, int meshSize, double[] refValues) {

    int index = -1;

    // minimum circularity and instance of time
    {    
        string[] qNameHints = new string[] { "circularity" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues circData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                circData = datGroups[i];
            }
        }

        (double circMin, double time) = GetMinimumCircularity(circData);
        Console.WriteLine($"Minimum circularity: {circMin} at time {time}");
        NUnit.Framework.Assert.AreEqual(refValues[0], circMin, 1e-4, 
            $"Minimum circularity does not match reference value {refValues[0]}.");
        NUnit.Framework.Assert.AreEqual(refValues[1], time, 1e-4, 
            $"Time for minimum circularity does not match reference value {refValues[1]}.");
    }

    // Maximum rise velocity and instance of time
    {    
        string[] qNameHints = new string[] { "rise velocity" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues riseVelData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                riseVelData = datGroups[i];
            }
        }

        (double riseVelMax, double time) = GetMaximumRiseVelocity(riseVelData);
        Console.WriteLine($"Maximum rise velocity: {riseVelMax} at time {time}");
        NUnit.Framework.Assert.AreEqual(refValues[2], riseVelMax, 1e-4, 
            $"Maximum rise velocity does not match reference value {refValues[2]}.");
        NUnit.Framework.Assert.AreEqual(refValues[3], time, 1e-4, 
            $"Time for maximum rise velocity does not match reference value {refValues[3]}.");
    }

    // Terminal rise height
    {    
        string[] qNameHints = new string[] { "center of mass - y" };
        for (int i = 0; i < data.Count(); i++) {
            if (qNameHints.Contains(data.ElementAt(i).Ylabel))
                index = i;
        }
        
        var datGroups = data.ElementAt(index).dataGroups;
        Plot2Ddata.XYvalues heightData = null;
        for (int i = 0; i < datGroups.Count(); i++) {
            if (datGroups[i].Name.Contains($"{study}-ConvStudy-k{pDeg}-mesh{meshSize}")) {
                heightData = datGroups[i];
            }
        }

        double height = GetTerminalRiseHeight(heightData);
        Console.WriteLine($"Terminal rise height: {height}");
        NUnit.Framework.Assert.AreEqual(refValues[4], height, 5e-4, 
            $"Terminal rise height does not match reference value {refValues[4]}.");
    }
}

In [32]:
CheckBenchmarkQuantities(evalData, "tc1", 2, 80, 
    new double[] { 0.9021, 1.8900, 0.2413, 0.9180, 1.0800});

Minimum circularity: 0.902127081240987 at time 1.88999999999993
Maximum rise velocity: 0.241348635611234 at time 0.918000000000025
Terminal rise height: 1.0803628075547


In [33]:
CheckBenchmarkQuantities(evalData, "tc1", 3, 60, 
    new double[] { 0.9025, 1.8870, 0.2412, 0.9180, 1.0800});

Minimum circularity: 0.9025299283894 at time 1.88699999999993
Maximum rise velocity: 0.241201457700652 at time 0.918000000000025
Terminal rise height: 1.08001207446236


## FIGURE 9 - Comparison with benchmark quantities

### Compare to Reference data (FEATFLOW benchmark)

In Hysing et al. an extensive dataset is provided by the following three research groups: 
TP2D (transport phenomena in 2D), FreeLIFE (free-surface library of finite element), and MoonMD (mathematics and object-oriented numerics in Magdeburg). 
For details on the methodology of each solver the reader is referred to the benchmark paper. 
The plotted data of the benchmark groups are taken from http://www.featflow.de/en/benchmarks/cfdbenchmarking/bubble/bubble_reference.html. 
Furthermore, we compare in the following some results with the work of Heimann et al., where the unfitted DG method with a piecewise linear approximation of the interface is employed.

In [20]:
string caseName = "testcase1";

In [21]:
static string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [22]:
// g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
static string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
static int[] datLvl;
static string datCase;
if(caseName.Equals("testcase1")) {
    datCase = "c1g";
    datLvl  = new int[] {7, 3, 4};    // testcase 1
} 
if(caseName.Equals("testcase2")) {     
    datCase = "c2g";
    datLvl  = new int[] {8, 3, 4};    // testcase 2
}

Plot2Ddata[,] dataRef = new Plot2Ddata[4,3];
for (int grp = 1; grp <= groups.Length; grp++) {
    Plot2Ddata[] datSet = new Plot2Ddata[4];
    // 1: area 2: circularity 3: center y 4: rise velocity
    for (int j = 0; j < 4; j++) {
        datSet[j] = new Plot2Ddata();
    }

    string dat  = datCase+grp+"l"+datLvl[grp - 1]+".txt";
    string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"Featflow_referenceData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] time = new double[lines.Length];
        double[][] valueDat = new double[4][];
        for(int j = 0; j < 4; j++)
            valueDat[j] = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            //var datString = lines[i].Split(new string[] {" "}, StringSplitOptions.RemoveEmptyEntries);
            //Console.WriteLine("num split strings at 0: {0}", datString[0]);
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            for (int j = 0; j < 4; j++) {
                valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            }
        }        
        // Build DataSet
        for (int j = 0; j < 4; j++) {
            string datName = groups[grp-1]+"-l"+datLvl[grp - 1];
            datSet[j] = (new Plot2Ddata(new KeyValuePair<string, double[][]>(datName, new double[][] { time, valueDat[j] })));
        }
    
    for (int j = 0; j < 4; j++) {
        dataRef[j,grp-1] = datSet[j];
    }
}

In [23]:
public static Plot2Ddata GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(List<Plot2Ddata> data, Plot2Ddata[,] refData, string[] qNameHints, string[] LegendAlign, double xMin, double xMax) {

    Plot2Ddata plt =  new Plot2Ddata();

    int index = -1;
    for (int i = 0; i < data.Count(); i++) {
        if (qNameHints.Contains(data.ElementAt(i).Ylabel))
            index = i;
    }
    
    int lcIdx = 1;
    var datGroups = data.ElementAt(index).dataGroups;
    for (int i = 0; i < datGroups.Count(); i++) {
        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData[3] == "k2" && nameData[4] == "mesh80"
            || nameData[3] == "k3" && nameData[4] == "mesh60") {

            var fmt = new PlotFormat();
            fmt.Style = Styles.Lines; 
            fmt.DashType = DashTypes.Solid;
            fmt.LineWidth = 2;
            fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

            plt.AddDataGroup($"{nameData[3]}-{nameData[4]}", datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    string[] refProps = new string[] { "area", "circularity", "center of mass - y", "rise velocity" };
    index = -1;
    for (int i = 0; i < refProps.Length; i++) {
        if (qNameHints.Contains(refProps[i]))
            index = i;
    }

    for (int i = 0; i < refData.GetLength(1); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.DotDashed;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx);
        
        // Add reference data
        plt.AddDataGroup(refData[index, i].dataGroups[0], fmt);
        lcIdx++;
    }

    plt.Xlabel = "Time";
    plt.Ylabel = qNameHints[0];

    plt.XrangeMin = xMin;
    plt.XrangeMax = xMax;
    // plt.YrangeMin = yMin;
    // plt.YrangeMax = yMax;

    plt.LegendAlignment = LegendAlign;
    plt.ShowLegend = LegendAlign != null ? true : false;
    
    return plt;
}

In [24]:
Plot2Ddata[,] Fig9 = new Plot2Ddata[3, 2];

Fig9[0, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"center of mass - y"}, new string[] { "i", "b", "r" }, 0.0, 3.0);
Fig9[0, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"center of mass - y"}, null, 2.7, 3.0);

Fig9[1, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"rise velocity"}, new string[] { "i", "b", "r" }, 0.0, 3.0);
Fig9[1, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"rise velocity"}, null, 0.8, 1.1);

Fig9[2, 0] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"circularity"}, new string[] { "i", "t", "r" }, 0.0, 3.0);
Fig9[2, 1] = GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(evalData, dataRef, new string[] {"circularity"}, null, 1.8, 2.0);

Fig9.ToGnuplot().PlotSVG(xRes:1200,yRes:1200)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 center of mass - y 
 
 
 
 
 Time 
 
 
 
 
 k2-mesh80 
 
 
 
 
 k2-mesh80 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M349.2,221.4 L402.6,221.4 M77.8,338.4 L78.3,338.4 L78.7,338.4 L79.2,338.4 L79.7,338.4 L80.1,338.4
 L80.6,338.4 L81.1,338.3 L81.5,338.3 L82.0,338.3 L82.5,338.3 L82.9,338.3 L83.4,338.2 L83.9,338.2
 L84.3,338.2 L84.8,338.1 L85.3,338.1 L85.7,338.1 L86.2,338.0 L86.7,338.0 L87.1,337.9 L87.6,337.9
 L88.1,337.8 L88.5,337.8 L89.0,337.7 L89.5,337.7 L89.9,337.6 L90.4,337.6 L90.9,337.5 L91.3,337.4
 L91.8,337.4 L92.3,337.3 L92.7,337.2 L93.2,337.2 L93.7,337.1 L94.1,337.0 L94.6,336.9 L95.1,336.9
 L95.5,336.8 L96.0,336.7 L96.5,336.6 L96.9,336.5 L97.4,336.4 L97.9,336.3 L98.3,336.2 L98.8,336.1
 L99.2,336.0 L99.7,335.9 L100.2,335.8 L100.6,335.7 L101.1,335.6 L101.6,335.5 L102.0,335.4 L102.5,335.3
 L103.0,335.2 L103.4,335.1 L103.9,334.9 L104.4,334.8 L104.8,334.7 L105.3,334.6 L105.8,334.4 L106.2,334.3
 L106.7,334.2 L107.2,334.0 L107.6,333.9 L108.1,333.8 L108.6,333.6 L109.0,333.5 L109.5,333.4 L110.0,333.2
 L110.4,333.1 L110.9,332.9 L111.4,332.8 L111.8,332.6 L112.3,332.5 L112.8,332.3 L113.2,332.2 L113.7,332.0
 L114.2,331.8 L114.6,331.7 L115.1,331.5 L115.6,331.3 L116.0,331.2 L116.5,331.0 L117.0,330.8 L117.4,330.7
 L117.9,330.5 L118.4,330.3 L118.8,330.1 L119.3,329.9 L119.8,329.8 L120.2,329.6 L120.7,329.4 L121.2,329.2
 L121.6,329.0 L122.1,328.8 L122.6,328.6 L123.0,328.4 L123.5,328.3 L124.0,328.1 L124.4,327.9 L124.9,327.7
 L125.4,327.5 L125.8,327.2 L126.3,327.0 L126.8,326.8 L127.2,326.6 L127.7,326.4 L128.2,326.2 L128.6,326.0
 L129.1,325.8 L129.6,325.6 L130.0,325.3 L130.5,325.1 L131.0,324.9 L131.4,324.7 L131.9,324.5 L132.4,324.2
 L132.8,324.0 L133.3,323.8 L133.8,323.5 L134.2,323.3 L134.7,323.1 L135.2,322.8 L135.6,322.6 L136.1,322.4
 L136.6,322.1 L137.0,321.9 L137.5,321.6 L138.0,321.4 L138.4,321.2 L138.9,320.9 L139.4,320.7 L139.8,320.4
 L140.3,320.2 L140.8,319.9 L141.2,319.6 L141.7,319.4 L142.1,319.1 L142.6,318.9 L143.1,318.6 L143.5,318.4
 L144.0,318.1 L144.5,317.8 L144.9,317.6 L145.4,317.3 L145.9,317.0 L146.3,316.8 L146.8,316.5 L147.3,316.2
 L147.7,316.0 L148.2,315.7 L148.7,315.4 L149.1,315.1 L149.6,314.8 L150.1,314.6 L150.5,314.3 L151.0,314.0
 L151.5,313.7 L151.9,313.4 L152.4,313.2 L152.9,312.9 L153.3,312.6 L153.8,312.3 L154.3,312.0 L154.7,311.7
 L155.2,311.4 L155.7,311.1 L156.1,310.8 L156.6,310.5 L157.1,310.2 L157.5,310.0 L158.0,309.7 L158.5,309.4
 L158.9,309.1 L159.4,308.8 L159.9,308.4 L160.3,308.1 L160.8,307.8 L161.3,307.5 L161.7,307.2 L162.2,306.9
 L162.7,306.6 L163.1,306.3 L163.6,306.0 L164.1,305.7 L164.5,305.4 L165.0,305.1 L165.5,304.7 L165.9,304.4
 L166.4,304.1 L166.9,303.8 L167.3,303.5 L167.8,303.2 L168.3,302.8 L168.7,302.5 L169.2,302.2 L169.7,301.9
 L170.1,301.6 L170.6,301.2 L171.1,300.9 L171.5,300.6 L172.0,300.3 L172.5,299.9 L172.9,299.6 L173.4,299.3
 L173.9,299.0 L174.3,298.6 L174.8,298.3 L175.3,298.0 L175.7,297.6 L176.2,297.3 L176.7,297.0 L177.1,296.6
 L177.6,296.3 L178.1,296.0 L178.5,295.6 L179.0,295.3 L179.5,295.0 L179.9,294.6 L180.4,294.3 L180.9,293.9
 L181.3,293.6 L181.8,293.3 L182.3,292.9 L182.7,292.6 L183.2,292.2 L183.7,291.9 L184.1,291.6 L184.6,291.2
 L185.0,290.9 L185.5,290.5 L186.0,290.2 L186.4,289.8 L186.9,289.5 L187.4,289.2 L187.8,288.8 L188.3,288.5
 L188.8,288.1 L189.2,287.8 L189.7,287.4 L190.2,287.1 L190.6,286.7 L191.1,286.4 L191.6,286.0 L192.0,285.7
 L192.5,285.3 L193.0,285.0 L193.4,284.6 L193.9,284.3 L194.4,283.9 L194

In [33]:
public static void CheckAgainstReferenceData(Plot2Ddata.XYvalues[] xyValues, Plot2Ddata.XYvalues[] refValues, double[] tolerance) {

    if (refValues.Length != tolerance.Length) {
        throw new ArgumentException("Reference values and tolerances arrays must have the same length.");
    }

    for (int j = 0; j < xyValues.Length; j++) {
        Console.WriteLine($"Checking data set {xyValues[j].Name}:");
        for (int i = 0; i < refValues.Length; i++) {    
            Dictionary<string, double> ret = ISessionInfoExtensions.ComputeErrorNormsForLogData(xyValues[j], refValues[i]);

            Console.WriteLine($"Reference data set {refValues[i].Name} - l1-error = {ret["l_1 error norm"]}");
            // check l1 norm against tolerance
            NUnit.Framework.Assert.IsTrue(ret["l_1 error norm"] <= tolerance[i], 
                $"L1 norm {ret["l_1 error norm"]} exceeds tolerance {tolerance[i]}.");
        }
    }
}

Checking center of mass

In [37]:
Plot2Ddata checkData = Fig9[0,0]; 
CheckAgainstReferenceData(
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[0],   // k2-mesh80
        checkData.dataGroups[1]    // k3-mesh60
    },
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[2],   // TP2D
        checkData.dataGroups[3],   // FreeLIFE
        checkData.dataGroups[4]    // MooNMD
    },
    new double[] {
        1.0e-3,
        1.0e-3,
        1.0e-3
    });

Checking data set k2-mesh80:
Reference data set TP2D-l7 - l1-error = 0.0004465672432818004
Reference data set FreeLIFE-l3 - l1-error = 0.000587620580059329
Reference data set MooNMD-l4 - l1-error = 0.0006799869570253317
Checking data set k3-mesh60:
Reference data set TP2D-l7 - l1-error = 0.0006247652563509675
Reference data set FreeLIFE-l3 - l1-error = 0.00038674298069945646
Reference data set MooNMD-l4 - l1-error = 0.0008805170390541179


Checking rise velocity

In [42]:
Plot2Ddata checkData = Fig9[1,0]; 
CheckAgainstReferenceData(
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[0],   // k2-mesh80
        checkData.dataGroups[1]    // k3-mesh60
    },
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[2],   // TP2D
        checkData.dataGroups[3],   // FreeLIFE
        checkData.dataGroups[4]    // MooNMD
    },
    new double[] {
        5.0e-3,
        1.0e-2,
        5.0e-3
    });

Checking data set k2-mesh80:
Reference data set TP2D-l7 - l1-error = 0.0024831749136524726
Reference data set FreeLIFE-l3 - l1-error = 0.004609056893929872
Reference data set MooNMD-l4 - l1-error = 0.002382261536347692
Checking data set k3-mesh60:
Reference data set TP2D-l7 - l1-error = 0.0030681060298280603
Reference data set FreeLIFE-l3 - l1-error = 0.005209869960814477
Reference data set MooNMD-l4 - l1-error = 0.0029529250366633155


Checking circularity

In [43]:
Plot2Ddata checkData = Fig9[2,0]; 
CheckAgainstReferenceData(
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[0],   // k2-mesh80
        checkData.dataGroups[1]    // k3-mesh60
    },
    new Plot2Ddata.XYvalues[] {
        checkData.dataGroups[2],   // TP2D
        checkData.dataGroups[3],   // FreeLIFE
        checkData.dataGroups[4]    // MooNMD
    },
    new double[] {
        1.0e-3,
        1.0e-3,
        1.0e-3
    });

Checking data set k2-mesh80:
Reference data set TP2D-l7 - l1-error = 0.0005257024431666661
Reference data set FreeLIFE-l3 - l1-error = 0.0006688890008808476
Reference data set MooNMD-l4 - l1-error = 0.0005322355191386749
Checking data set k3-mesh60:
Reference data set TP2D-l7 - l1-error = 0.0007254160947653302
Reference data set FreeLIFE-l3 - l1-error = 0.0008834963796045508
Reference data set MooNMD-l4 - l1-error = 0.0007260133504182341


## FIGURE B1 - Comparison terminal bubble shapes

In [44]:
public static (double[] x, double[] y) GetTerminalShapeInterfacePoints(ISessionInfo sess) {

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    return (x, y);
}

In [45]:
public static Plot2Ddata GetShapeComparison_Plot2Ddata(List<ISessionInfo> dataSess, string caseName, string[] study) {

    Plot2Ddata datShape = new Plot2Ddata();

    int lcIdx = 1;
    for(int i = 0; i < study.Length; i++) {
        foreach (var dataS in dataSess) {
            if (dataS.Name.Contains(study[i])) {
                var shape = GetTerminalShapeInterfacePoints(dataS);

                var fmt = new PlotFormat();
                fmt.Style = Styles.Lines;
                fmt.Style      = Styles.Points;
                fmt.PointType  = PointTypes.Dot;
                fmt.LineColor = (LineColors)(lcIdx);

                datShape.AddDataGroup((study[i]).Replace("_", "-"), shape.x, shape.y, fmt);
                lcIdx++;
            }
        }
    }

    // Add reference shapes
    // g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
    string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
    int[] datLvl = null;
    string datCase = "";
    if(caseName.Equals("testcase1")) {
        datCase = "c1g";
        datLvl  = new int[] {7, 3, 4};    // testcase 1
    } 
    if(caseName.Equals("testcase2")) {     
        datCase = "c2g";
        datLvl  = new int[] {8, 3, 4};    // testcase 2
    }

    for (int grp = 1; grp <= groups.Length; grp++) {

        string dat  = datCase+grp+"l"+datLvl[grp - 1]+"s.txt";
        string path = (userName.Equals(@"FDY\jenkinsci")) ? dat : @"Featflow_referenceData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] x = new double[lines.Length];
        double[] y = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            x[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            y[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[1]);      
        }   

        string datName = groups[grp-1]+"-l"+datLvl[grp - 1]+"s";

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.Style      = Styles.Points;
        fmt.PointType  = PointTypes.Dot;
        fmt.LineColor = (LineColors)(9 - grp);

        datShape.AddDataGroup(datName, x, y , fmt);
    }

    datShape.LegendAlignment = new string[] { "o", "t", "r" };

    return datShape;
}

In [46]:
Plot2Ddata shapePlt = GetShapeComparison_Plot2Ddata(evalSess, caseName, new string[] { "k2_mesh80", "k3_mesh60" });

shapePlt.ToGnuplot().PlotSVG(xRes:1200,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.95 
 
 
 
 
 1 
 
 
 
 
 1.05 
 
 
 
 
 1.1 
 
 
 
 
 1.15 
 
 
 
 
 1.2 
 
 
 
 
 1.25 
 
 
 
 
 1.3 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 k2-mesh80 
 
 
 k2-mesh80 
 
 
 
 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

In [47]:
public static void CheckInterfaceShapeAgainstReferenceData(Plot2Ddata.XYvalues xyValues, Plot2Ddata.XYvalues refValues, double tolerance) {

    double distError = 0.0;
    for (int i = 0; i < refValues.Values.Length; i++) {
        double minDist = double.MaxValue;
        for (int j = 0; j < xyValues.Values.Length; j++) {
            double dist = Math.Sqrt( (refValues.Abscissas[i] - xyValues.Abscissas[j]).Pow2() + (refValues.Values[i] - xyValues.Values[j]).Pow2() );
            if (dist < minDist)
                minDist = dist;
        }
        distError += minDist;
    }
    distError /= refValues.Values.Length;

    // check l1 norm against tolerance
    NUnit.Framework.Assert.IsTrue(distError <= tolerance, 
        $"Anbsolute distance error {distError} of interface shape exceeds tolerance {tolerance}.");
}

Checking against TP2D

In [54]:
for (int i = 0; i < 2; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[i], shapePlt.dataGroups[2], 2e-3);
}

Checking data group k2-mesh80 against reference data.
Checking data group k3-mesh60 against reference data.


Checking against FreeLIFE

In [57]:
for (int i = 0; i < 2; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[i], shapePlt.dataGroups[3], 1e-3);
}

Checking data group k2-mesh80 against reference data.
Checking data group k3-mesh60 against reference data.


Checking against MooNMD

In [60]:
for (int i = 0; i < 2; i++) {
    Console.WriteLine($"Checking data group {shapePlt.dataGroups[i].Name} against reference data.");
    CheckInterfaceShapeAgainstReferenceData(shapePlt.dataGroups[i], shapePlt.dataGroups[4], 2e-3);
}

Checking data group k2-mesh80 against reference data.
Checking data group k3-mesh60 against reference data.
